In [1]:
from pyspark.sql import functions as F

silver_caged = (
    spark.table("silver_caged")
    .select(
        "ano",
        "mes",
        "cnae_2_descricao_secao",
        "idade",
        "salario_mensal",
        "saldo_movimentacao",
        F.when(F.col("saldo_movimentacao") == -1, "Demissões")
         .otherwise("Admissões")
         .alias("variable")
    )
)


silver_caged = silver_caged.withColumn(
    "ano_mes",
    F.make_date(
        F.col("ano").cast("int"),
        F.col("mes").cast("int"),
        F.lit(1)
    )
)

df_gold_caged_saldo_movimentacao_anual = silver_caged.groupBy(
    "ano", "mes", "ano_mes", "cnae_2_descricao_secao"
).agg(
    F.sum("saldo_movimentacao").alias("saldo_movimentacao"),
    F.round(F.avg("salario_mensal"), 2).alias("salario_mensal"),
)

(
    df_gold_caged_saldo_movimentacao_anual.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_caged_saldo_movimentacao_anual")
)

# SALDO SEÇÃO

df_gold_caged_saldo_secao = silver_caged.groupBy("ano", "mes", "ano_mes", "cnae_2_descricao_secao").agg(
    F.sum("saldo_movimentacao").alias("saldo_movimentacao"),
    F.sum("salario_mensal").alias("salario_mensal"),
)


(
    df_gold_caged_saldo_secao.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_caged_saldo_secao")
)

# MÉDIA IDADE

df_gold_caged_media_idade = silver_caged.groupBy(
    "ano", "mes", "ano_mes", "cnae_2_descricao_secao", "variable"
).agg(F.round(F.avg("idade"), 2).alias("media_idade"))

(
    df_gold_caged_media_idade.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_caged_media_idade")
)

# MÉDIA SALARIO
df_gold_caged_media_salario = silver_caged.groupBy(
    "ano", "mes", "ano_mes", "cnae_2_descricao_secao", "variable"
).agg(F.round(F.avg("salario_mensal"), 2).alias("salario_medio"))

(
    df_gold_caged_media_salario.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_caged_media_salario")
)

# SALDO IDADE

df_faixa_etaria = silver_caged.select("ano", "mes", "ano_mes", "saldo_movimentacao", "idade").withColumn(
    "idade_grupo",
    F.when(F.col("idade") <= 5, "0-5")
    .when((F.col("idade") >= 6) & (F.col("idade") <= 10), "6-10")
    .when((F.col("idade") >= 11) & (F.col("idade") <= 15), "11-15")
    .when((F.col("idade") >= 16) & (F.col("idade") <= 20), "16-20")
    .when((F.col("idade") >= 21) & (F.col("idade") <= 25), "21-25")
    .when((F.col("idade") >= 26) & (F.col("idade") <= 30), "26-30")
    .when((F.col("idade") >= 31) & (F.col("idade") <= 35), "31-35")
    .when((F.col("idade") >= 36) & (F.col("idade") <= 40), "36-40")
    .when((F.col("idade") >= 41) & (F.col("idade") <= 45), "41-45")
    .when((F.col("idade") >= 46) & (F.col("idade") <= 50), "46-50")
    .when((F.col("idade") >= 51) & (F.col("idade") <= 55), "51-55")
    .when((F.col("idade") >= 56) & (F.col("idade") <= 60), "56-60")
    .when((F.col("idade") >= 61) & (F.col("idade") <= 65), "61-65")
    .when((F.col("idade") >= 66) & (F.col("idade") <= 70), "66-70")
    .when((F.col("idade") >= 71) & (F.col("idade") <= 75), "71-75")
    .when((F.col("idade") >= 76) & (F.col("idade") <= 80), "76-80")
    .when(F.col("idade") > 80, "80+")
    .otherwise("N/A"),
)

df_gold_caged_saldo_idade = df_faixa_etaria.groupBy("ano", "mes", "ano_mes", "idade_grupo").agg(
    F.sum("saldo_movimentacao").alias("saldo_movimentacao")
)

(
    df_gold_caged_saldo_idade.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_caged_saldo_idade")
)


StatementMeta(, 86042546-0d09-488b-9cba-0516184abc29, 3, Finished, Available, Finished, True)